In [0]:
# Databricks notebook source
"""
HealthFlow Data Quality & Integrity Test Suite
Validates schema completeness, primary key uniqueness, and cross-table referential integrity.
"""

from pyspark.sql.functions import col

print("🧪 Starting HealthFlow Data Quality Suite...\n")

# -------------------------------------------------------------
# 1. READ SILVER TABLES
# -------------------------------------------------------------
df_patients = spark.table("workspace.silver.patients")
df_encounters = spark.table("workspace.silver.encounters")
df_conditions = spark.table("workspace.silver.conditions")
df_procedures = spark.table("workspace.silver.procedures")
df_claims = spark.table("workspace.silver.claims")

# -------------------------------------------------------------
# 2. CHECK 1: NULL PRIMARY KEYS
# -------------------------------------------------------------
null_checks = {
    "patients": df_patients.filter(col("patient_id").isNull()).count(),
    "encounters": df_encounters.filter(col("encounter_id").isNull()).count(),
    "conditions": df_conditions.filter(col("patient_id").isNull()).count(),
    "procedures": df_procedures.filter(col("patient_id").isNull()).count(),
    "claims": df_claims.filter(col("claim_id").isNull()).count(),
}

for table, null_count in null_checks.items():
    assert null_count == 0, f"❌ DQ Failure: Found {null_count} null primary keys in {table}!"
    print(f"✅ {table.capitalize()} Null Key Check Passed (0 nulls)")

# -------------------------------------------------------------
# 3. CHECK 2: PRIMARY KEY UNIQUENESS
# -------------------------------------------------------------
patient_dups = df_patients.groupBy("patient_id").count().filter(col("count") > 1).count()
encounter_dups = df_encounters.groupBy("encounter_id").count().filter(col("count") > 1).count()

assert patient_dups == 0, f"❌ DQ Failure: Found duplicate patient_ids!"
assert encounter_dups == 0, f"❌ DQ Failure: Found duplicate encounter_ids!"

print("✅ Primary Key Uniqueness Checks Passed")

# -------------------------------------------------------------
# 4. CHECK 3: REFERENTIAL INTEGRITY (Orphan Records)
# -------------------------------------------------------------
# Encounters with non-existent Patient IDs
orphan_encounters = (
    df_encounters
    .join(df_patients, "patient_id", "left_anti")
    .count()
)

assert orphan_encounters == 0, f"❌ DQ Failure: Found {orphan_encounters} encounters referencing non-existent patients!"
print(f"✅ Referential Integrity Passed (0 orphan encounters)")

# -------------------------------------------------------------
# 5. CHECK 4: BUSINESS LOGIC / VALUE BOUNDS
# -------------------------------------------------------------
negative_costs = df_claims.filter(col("total_cost") < 0).count()
assert negative_costs == 0, f"❌ DQ Failure: Found {negative_costs} claims with negative cost!"
print("✅ Business Value Bounds Check Passed")

print("\n🎉 ALL DATA QUALITY TESTS PASSED SUCCESSFULLY!")